In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy 
from pfapack import pfaffian as pf


We assume the form 

$$X = e^{\frac{1}{4}\gamma^T A \gamma}$$

which is characterized by 

$$\eta, B$$

In [2]:
def get_eta_B(A):

    B = scipy.linalg.expm(-A)

    sinhA = scipy.linalg.sinhm(A/4)

    bdim = B.shape[0]

    M = np.zeros((2*bdim, 2*bdim), dtype = np.complex128)

    M[0:bdim, 0:bdim] = np.sqrt(2) * sinhA
    M[bdim:,0:bdim]   = np.identity(bdim)
    M[0:bdim,bdim:]   = -np.identity(bdim)
    M[bdim:,bdim:]    = np.sqrt(2) * sinhA

    eta = np.power(-1,bdim//2) * pf.pfaffian(M)

    return eta, B

def multiplication(eta1, B1, eta2, B2):

    bdim = B1.shape[0]

    M = np.zeros((2*bdim, 2*bdim), dtype = np.complex128)

    G1 = (np.identity(bdim) - B1) @ scipy.linalg.inv(np.identity(bdim) + B1)
    G2 = (np.identity(bdim) - B2) @ scipy.linalg.inv(np.identity(bdim) + B2)

    M[0:bdim, 0:bdim] = G1
    M[bdim:,0:bdim]   = np.identity(bdim)
    M[0:bdim,bdim:]   = -np.identity(bdim)
    M[bdim:,bdim:]    = G2

    eta12 = np.power(-1,bdim//2) * eta1 * eta2 * pf.pfaffian(M)

    B12   = B1@B2

    return eta12, B12

def recursive_eta(Alist):

    Ntau = len(Alist)
    N    = Alist[0].shape[0]//2

    eta, B = get_eta_B(Alist[0])

    eta_prod = eta
    B_prod = B

    for i in range(Ntau-1):

        next_eta, next_B = get_eta_B(Alist[i+1])

        eta_prod, B_prod = multiplication(eta_prod, B_prod, next_eta, next_B)

    return eta_prod*np.power(2, N)

def brute_force_trace(Alist):

    Ntau = len(Alist)
    N    = Alist[0].shape[0] // 2


    eta_prod = 1
    etas = np.zeros(Ntau, dtype=np.complex128)
    Bs   = np.zeros((2*N, 2*N, Ntau), dtype=np.complex128)
    Gs   = np.zeros((2*N, 2*N, Ntau), dtype=np.complex128)

    M    = np.zeros((2*N*Ntau, 2*N*Ntau), dtype=np.complex128)

    I = np.eye(2*N, dtype=np.complex128)

    # Fill M in 2N x 2N blocks
    for i in range(Ntau):
        for j in range(i + 1, Ntau):
            # checkerboard on the upper-right triangle, starting with "-"
            sgn = -1 if ((i + j) % 2 == 1) else 1

            r_i = slice(2*N*i, 2*N*(i + 1))
            r_j = slice(2*N*j, 2*N*(j + 1))

            M[r_i, r_j] = sgn * I          # upper-right block
            M[r_j, r_i] = -sgn * I         # lower-left block (antisymmetric)


    for i in range(Ntau):
        etas[i], Bs[:, :, i] = get_eta_B(Alist[i])
        Gs[:, :, i]          = scipy.linalg.tanhm(Alist[i] / 2)
        eta_prod *= etas[i]
        M[i*2*N:(i+1)*2*N,i*2*N:(i+1)*2*N] = Gs[:,:,i]

    W = np.power(-1, N*Ntau//2) * np.power(2, N) * eta_prod * pf.pfaffian(M)

    return W

def catterpillar(Alist):

    Ntau = len(Alist)
    N    = Alist[0].shape[0] // 2

    temp = np.zeros((4*N, 4*N), dtype = np.complex128)

    Bs = np.zeros((2*N, 2*N, Ntau), dtype = np.complex128)
    etas = np.zeros(Ntau, dtype = np.complex128)
    eta_prod = 1
    pf_prod  = 1

    current_Bstring = np.identity(2*N, dtype=np.complex128)

    for i in range(Ntau):

        etas[i], Bs[:,:,i] = get_eta_B(Alist[i])
        eta_prod *= etas[i]

    for i in range(Ntau - 1):

        current_Bstring = current_Bstring @ Bs[:,:,i]

        temp[:2*N,:2*N] = (np.identity(2*N) - current_Bstring) @ scipy.linalg.inv(np.identity(2*N) + current_Bstring)
        temp[:2*N,2*N:] = -np.identity(2*N)
        temp[2*N:,:2*N] = np.identity(2*N)
        temp[2*N:,2*N:] = scipy.linalg.tanhm(0.5*Alist[i+1])

        pf_prod *= pf.pfaffian(temp)

    W = np.power(2, N) * np.power(-1, (Ntau + 1)*N) *eta_prod * pf_prod

    return W

def generate_random_Alist(N, Ntau, scaling = 0.1):

    Alist = []
    rng = np.random.default_rng(seed = 412425)

    for i in range(Ntau):

        A = rng.normal(loc = 0, scale = scaling, size = (2*N, 2*N)) + 1j * rng.normal(loc = 0, scale = scaling, size = (2*N, 2*N))
        A = 0.5*(A - A.T)
        Alist.append(A)

    return Alist

def get_J_cross(matrix):

    dim = matrix.shape[0]

    Jcross = np.zeros((2*dim, 2*dim), dtype = matrix.dtype)

    Jcross[:dim, dim:2*dim] = - matrix
    Jcross[dim:2*dim, :dim] = + matrix

    return Jcross

def cos_string(phi,dtau):

    prod = 1

    for i in range(phi.size):

        prod = prod * np.cos(phi[i]*dtau) * (-1)

    return prod


class HMC():

    def __init__(self, t_MD, N_MD, model, saver, seed = 1337):
        self.N_MD = N_MD
        self.t_MD  = t_MD
        self.dt, self.dt_half = t_MD/self.N_MD, 0.5*t_MD/self.N_MD
        self.model = model
        self.cfg_shape = model.shape
        self.saver = saver
        self.steps = self.acc = 0
        self.rng   = np.random.default_rng(seed)


    def step(self):

        self.mom = self.rng.normal(loc = 0.0, scale = 1.0, size = self.cfg_shape).astype(np.complex128)

        new_cfg  = np.copy(self.model.cfg)

        old_H = self.model.action + 0.5 * np.sum(np.power(self.mom, 2))

        self.mom -= self.dt_half*self.model.get_action_gradient(new_cfg)

        for _ in range(self.N_MD-1):

            new_cfg +=self.dt*self.mom
            self.mom -= self.dt*self.model.get_action_gradient(new_cfg)
        
        new_cfg += self.dt*self.mom
        self.mom -= self.dt_half*self.model.get_action_gradient(new_cfg)

        new_action = self.model.get_action_catterpiller(new_cfg)

        new_H = new_action + 0.5* np.sum(np.power(self.mom, 2))


        dH    = new_H - old_H
        p = np.exp(-dH)
        r = self.rng.random()
        
        if r < p :

            self.acc += 1

            self.model.cfg = new_cfg
            self.model.action = new_action

        self.steps += 1


    def save(self):
        self.model.update_obs()
        self.saver.save_cfg_and_obs(self.model.cfg, self.model.obs_dict)

    def output_acc(self):
        return self.acc/self.steps
        
    def reset_acc(self):
        self.acc = self.steps = 0

    def output_model(self):
        return self.model

    def close_file(self):
        self.saver.close()



class Kitaev_action():
    def __init__(self, L, U, Delta, mu, dtau, Ntau, seed = 153525):
        #set paramters and initialize configuration     
        self.U = U

        self.mu = mu
        self.dtau = dtau
        self.Ntau = Ntau
        self.Delta = Delta
        self.L = L
        self.Dplus = 1 + Delta
        self.Dminus = 1 - Delta

        self.rng = np.random.default_rng(seed)
        self.shape = (L, Ntau)
        self.cfg = self.rng.normal(loc=0.0, scale = 1/np.sqrt(np.prod(self.shape)), size = self.shape).astype(np.complex128)
        self.gradient = None
        self.J = np.array([[0,-1],[1,0]])
        self.I = np.eye(2)

        self.A_odd = np.zeros((2*L, 2*L), dtype = np.complex128)
        self.A_odd[:L,:L] = np.roll(np.eye(L)*self.Dminus, 1, axis = 1) - np.roll(np.eye(L)*self.Dminus, -1, axis = 1)
        self.A_odd[L:,L:] = np.roll(np.eye(L)*self.Dplus, 1, axis = 1) - np.roll(np.eye(L)*self.Dplus, -1, axis = 1)

        self.A_odd[:L,L:] = -np.eye(L)*mu
        self.A_odd[L:,:L] = np.eye(L)*mu

        self.A_odd[L-1, 0] = 0
        self.A_odd[0, L-1] = 0

        self.A_odd[2*L - 1, L] = 0
        self.A_odd[L, 2*L-1]   = 0

        self.A_odd = self.A_odd * self.dtau * 1j
        self.eta_odd, self.B_odd = get_eta_B(self.A_odd)


        self.G_odd = scipy.linalg.tanhm(self.A_odd/2)
        self.expminusA = scipy.linalg.expm(-self.A_odd)
        self.eta_odd_prod = np.power(self.eta_odd, self.Ntau)

        self.temp = np.zeros((4*self.L, 4*self.L), dtype = np.complex128)
        self.temp[2*self.L:, :2*self.L] = np.eye(2*self.L)
        self.temp[:2*self.L, 2*self.L:] = -np.eye(2*self.L)

        self.g = np.zeros((self.Ntau*self.L*4, self.Ntau*self.L*4))

        block_size = 2 * L
        total_size = block_size * Ntau *2

        self.g = np.zeros((total_size, total_size), dtype=np.complex128)
        I = np.eye(block_size, dtype=np.complex128)

        for i in range(2*Ntau):
            for j in range(i + 1, 2*Ntau):
                # Checkerboard sign on upper triangle:
                # starts with minus for (i,j) = (0,1)
                sign = -1 if (j - i) % 2 == 1 else 1

                row_i = slice(i * block_size, (i + 1) * block_size)
                row_j = slice(j * block_size, (j + 1) * block_size)

                self.g[row_i, row_j] = sign * I
                self.g[row_j, row_i] = -sign * I  # antisymmetry

            

        self.action = self.get_action_catterpiller(self.cfg)

    def get_all_As(self, cfg):

        allAs = np.zeros((2*self.Ntau,self.L*2, self.L*2), dtype = np.complex128)

        for i in range(self.Ntau):

            allAs[2*i,:,:] = self.A_odd.copy()
            allAs[2*i+1,:,:] = np.kron(self.J, np.diag(2*self.dtau*cfg[:,i]))


        return allAs
    
    def get_g(self, cfg):

        block_size = 2*self.L
        I = np.eye(block_size, dtype=np.complex128)

        for j in range(self.Ntau):

                row_j = slice(2*j * block_size, (2*j + 1) * block_size)

                self.g[row_j, row_j] = self.G_odd
                
                #even case
                row_j = slice((2*j+1) * block_size, ((2*j+1) + 1) * block_size)
                self.g[row_j, row_j] = np.kron(self.J, np.diag(np.tan(self.dtau * cfg[:,j])))

                # plt.imshow(np.abs(self.g[row_j, row_j]))
                # plt.show()

    def get_action_catterpiller(self, cfg):

        allBs   = np.zeros((2*self.L, 2*self.L, self.Ntau), dtype = np.complex128)
        alletas = np.zeros(self.Ntau, dtype = np.complex128)
        allGs   = np.zeros((2*self.L, 2*self.L, self.Ntau), dtype = np.complex128)

        current_G = self.G_odd.copy()
        current_B = self.expminusA.copy()
        current_eta = self.eta_odd

        I2 = np.eye(2*self.L, dtype=np.complex128)

        for i in range(self.Ntau):
            
            allBs[:,:,i]   = np.kron(self.I, np.diag(np.cos(2 * self.dtau * cfg[:,i]))) - np.kron(self.J, np.diag(np.sin(2*self.dtau*cfg[:,i])))
            alletas[i] = cos_string(cfg[:,i], self.dtau)
            allGs[:,:,i]   = np.kron(self.J, np.diag(np.tan(self.dtau * cfg[:,i])))

            self.temp[:2*self.L, :2*self.L] = current_G
            self.temp[2*self.L:, 2*self.L:] = allGs[:,:,i]
            current_eta                     = np.power(-1, self.L) * current_eta * alletas[i] * pf.pfaffian(self.temp)
            current_B                       = current_B @ allBs[:,:,i]
            current_G                       = (I2 - current_B)@scipy.linalg.inv(I2 + current_B)

            # print("orth err =", np.linalg.norm(current_B.T @ current_B - I2))
            # print("skew err =", np.linalg.norm(current_G + current_G.T))
            # print("cond(I+B) =", np.linalg.cond(I2 + current_B))
            # print("min |eig(B)+1| =", np.min(np.abs(np.linalg.eigvals(current_B) + 1)))

            if i != (self.Ntau - 1):
                self.temp[:2*self.L, :2*self.L] = current_G
                self.temp[2*self.L:, 2*self.L:] = self.G_odd
                current_eta                     = np.power(-1, self.L) * current_eta * self.eta_odd * pf.pfaffian(self.temp)
                current_B                       = current_B @ self.expminusA
                current_G                       = (I2 - current_B)@scipy.linalg.inv(I2 + current_B)

            # print("orth err =", np.linalg.norm(current_B.T @ current_B - I2))
            # print("skew err =", np.linalg.norm(current_G + current_G.T))
            # print("cond(I+B) =", np.linalg.cond(I2 + current_B))
            # print("min |eig(B)+1| =", np.min(np.abs(np.linalg.eigvals(current_B) + 1)))

        weight = 2**self.L * current_eta

        action = np.log(weight)

        return weight
    
    def get_action_gradient(self, cfg):

        # W = self.get_action_catterpiller(cfg)
        gradient_p1 = -self.dtau * np.tan(self.dtau * cfg)

        self.get_g(cfg)

        g_inv = scipy.linalg.inv(self.g)

        t = np.arange(self.Ntau)[:, None]
        x = np.arange(self.L)[None, :]

        rows = (2*t + 1) * (2*self.L) + x
        cols = rows + self.L

        g_inv_diag = g_inv[rows, cols].reshape(self.Ntau, self.L)

        gradient_p2 = self.dtau * g_inv_diag.T*(1 + np.power(np.tan(self.dtau*cfg), 2))

        self.gradient = (gradient_p1 + gradient_p2)

        return self.gradient



In [3]:
L = 4
U = 1
Delta = 0.2
mu = 2
dtau = 0.1
Ntau = 2

model = Kitaev_action(L, U, Delta, mu, dtau, Ntau)
maction = model.get_action_catterpiller(model.cfg)

allAs = model.get_all_As(model.cfg)
baction = recursive_eta(allAs)

model.get_action_gradient(model.cfg)

print(maction)
print(baction)


(17.795025055268702+0.2342633519850633j)
(17.795025055268702+0.2342633519850633j)


In [10]:
L = 6
U = 1
Delta = 0.2
mu = 2
dtau = 0.1
Ntau = 2
t_MD = 0.3
N_MD = 8


model = Kitaev_action(L, U, Delta, mu, dtau, Ntau)
algorithm = HMC(t_MD, N_MD, model, saver = 0, seed = 3432)

for i in range(1000):
    algorithm.step()

algorithm.reset_acc()

for i in range(1000):
    algorithm.step()
algorithm.output_acc()

0.24

In [ ]:
N = 4
dtau = 0.2
rng = np.random.default_rng(4135)
phi = rng.normal(loc = 0.0, scale = 1, size = (N))
temp = np.diag(2*dtau*phi)

A = get_J_cross(temp)

eta, B = get_eta_B(A)
eta_fast = cos_string(phi, dtau)

print(eta)
print(eta_fast)


In [ ]:
catterpillar(Alist)

In [ ]:
brute_force_trace(Alist)

In [ ]:
recursive_eta(Alist)